# 01 数据概览与数据字典

**要回答的问题**：这份数据有多少行多少列？每一列是什么类型、什么含义？有没有缺失值？

**为什么先做这一步**：后面所有的图、所有的检验，都建立在"这一行代表什么"这个前提上。
前提没搞清楚，图画得再漂亮也可能是错的。

**方法**：读入 → 看前几行确认读对了 → `info()` 体检 → 逐列列出 dtype 与取值概况 → 检查缺失值。

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
from pathlib import Path
import pandas as pd

DATA = Path.cwd().parent / 'data' / 'raw' / 'smart_logistics_dataset.csv'
df = pd.read_csv(DATA, keep_default_na=False, na_values=[''])

In [3]:
df.head()

,Timestamp,Asset_ID,Latitude,Longitude,Inventory_Level,Shipment_Status,Temperature,Humidity,Traffic_Status,Waiting_Time,User_Transaction_Amount,User_Purchase_Frequency,Logistics_Delay_Reason,Asset_Utilization,Demand_Forecast,Logistics_Delay
0,2024/3/20 0:11,Truck_7,-65.7383,11.2497,390,Delayed,27.0,67.8,Detour,38,320,4,None,60.1,285,1
1,2024/10/30 7:53,Truck_6,22.2748,-131.7086,491,In Transit,22.5,54.3,Heavy,16,439,7,Weather,80.9,174,1
2,2024/7/29 18:42,Truck_10,54.9232,79.5455,190,In Transit,25.2,62.2,Detour,34,355,3,None,99.2,260,0
3,2024/10/28 0:50,Truck_9,42.3900,-1.4788,330,Delivered,25.4,52.3,Heavy,37,227,5,Traffic,97.4,160,1
4,2024/9/27 15:52,Truck_7,-65.8477,47.9468,480,Delayed,20.5,57.2,Clear,56,197,6,None,71.6,270,1


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Timestamp                1000 non-null   str    
 1   Asset_ID                 1000 non-null   str    
 2   Latitude                 1000 non-null   float64
 3   Longitude                1000 non-null   float64
 4   Inventory_Level          1000 non-null   int64  
 5   Shipment_Status          1000 non-null   str    
 6   Temperature              1000 non-null   float64
 7   Humidity                 1000 non-null   float64
 8   Traffic_Status           1000 non-null   str    
 9   Waiting_Time             1000 non-null   int64  
 10  User_Transaction_Amount  1000 non-null   int64  
 11  User_Purchase_Frequency  1000 non-null   int64  
 12  Logistics_Delay_Reason   1000 non-null   str    
 13  Asset_Utilization        1000 non-null   float64
 14  Demand_Forecast          1000 non-nu

## 缺失值检查

**要区分两件事**：真正的空缺（NaN）和"填了 `None` 这个文本"。读取时用了
`keep_default_na=False`，所以字符串 `None` 会被当作**有效取值**保留下来——这一点很关键，
否则 `dropna()` 会把有效数据误删。

In [5]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning,module='seaborn.categorical')

In [6]:
total_shipments = len(df)#返回df的行数
df['Logistics_Delay'] = df['Logistics_Delay'].astype('category')

In [7]:
# --- 数据字典：每列的 dtype 与取值概况 ---
data_dictionary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'non_null': df.notna().sum(),
    'n_unique': df.nunique(),
    'sample': [df[c].iloc[0] for c in df.columns],
})
data_dictionary

,dtype,non_null,n_unique,sample
Timestamp,str,1000,1000,2024/3/20 0:11
Asset_ID,str,1000,10,Truck_7
Latitude,float64,1000,1000,-65.7383
Longitude,float64,1000,1000,11.2497
Inventory_Level,int64,1000,366,390
Shipment_Status,str,1000,3,Delayed
Temperature,float64,1000,121,27.0
Humidity,float64,1000,291,67.8
Traffic_Status,str,1000,3,Detour
Waiting_Time,int64,1000,51,38


In [8]:
# --- 缺失值检查 ---
# 注意：读取时用了 keep_default_na=False，所以字符串 'None' 不会被当成缺失值。
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
})
print('真正缺失的单元格总数：', int(df.isna().sum().sum()))
missing[missing['missing_count'] > 0] if missing['missing_count'].sum() else \
    print('没有任何一列存在真正的缺失值（NaN）。')

# 关键区分：'None' 是有效取值，不是缺失值
print()
print("Logistics_Delay_Reason 取值分布：")
print(df['Logistics_Delay_Reason'].value_counts(dropna=False))

真正缺失的单元格总数： 0
没有任何一列存在真正的缺失值（NaN）。

Logistics_Delay_Reason 取值分布：
Logistics_Delay_Reason
Weather               267
None                  263
Traffic               236
Mechanical Failure    234
Name: count, dtype: int64


In [9]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Latitude,1000.0,-1.360093,51.997183,-89.7915,-46.167975,-4.50315,44.50280,89.8701
Longitude,1000.0,0.837049,104.843618,-179.8202,-88.448075,0.67830,88.15645,179.9237
Inventory_Level,1000.0,297.915000,113.554773,100.0000,201.000000,299.00000,399.00000,500.0000
Temperature,1000.0,23.893900,3.322178,18.0000,21.200000,23.80000,26.60000,30.0000
Humidity,1000.0,65.042200,8.753765,50.0000,57.200000,65.20000,72.40000,80.0000
Waiting_Time,1000.0,35.062000,14.477768,10.0000,23.000000,35.00000,49.00000,60.0000
User_Transaction_Amount,1000.0,299.055000,117.787792,100.0000,191.750000,301.50000,405.00000,500.0000
User_Purchase_Frequency,1000.0,5.513000,2.935379,1.0000,3.000000,6.00000,8.00000,10.0000
Asset_Utilization,1000.0,79.599100,11.631153,60.0000,69.475000,79.25000,89.42500,100.0000
Demand_Forecast,1000.0,199.284000,59.920847,100.0000,144.000000,202.00000,251.25000,300.0000


## 数值字段的描述统计

`describe().T` 把每个数值字段的样本量、均值、标准差、四分位数一次列全，
用来快速判断取值区间是否合理。